# **Grouped split: About the Dataset, Training and Experiments**

Code for the **grouped 70/20/10 split** of my honours thesis *Lightweight Object Detection for African Penguin Population Monitoring*  
(Hluma Xako, Rhodes University, 2026). It produces Experiments 1.2 to 5 and the figures in Chapter 4 and the Appendix.

| Section | Thesis experiment | Result file |
|---|---|---|
| 1. Build the dataset | Dataset (Chapter 3) | `penguin_dataset_702010/` |
| 2. Training | Four models, identical settings | `runs_702010/` |
| 3. Detection accuracy | Experiment 1.2 | `detection_702010.csv` |
| 4. Counting accuracy | Experiment 1.2 | `counting_702010.csv` |
| 5. CPU latency | Experiment 2 | `latency_702010.csv` |
| 6. FP32 against FP16 | Experiment 3 | `fp16_702010.csv` |
| 7. Hyperparameter tuning | Experiment 4.2 | Ray Tune results folder |
| 8. Scale augmentation | Experiment 5 | `scale_702010.csv` |
| 9. Figures and density split | Chapter 4 and Appendix | `thesis_figures/`, `density_702010.csv` |

Paths point to the machine the experiments ran on. Training and the latency timing should run with no background processes using the machine.

## **Setup**

In [ ]:
# !pip install ultralytics openvino imagehash opencv-python

import os, glob, time, csv, random, shutil
from pathlib import Path

import cv2
import numpy as np
import torch
import imagehash
import ultralytics
from PIL import Image
from ultralytics import YOLO

dataset_dir = "/home/hluma/penguin_dataset_702010"
data_yaml = dataset_dir + "/data.yaml"
african_yaml = dataset_dir + "/data_test_african.yaml"
results_dir = "/home/hluma/hluma_results"
runs_dir = results_dir + "/runs_702010"
os.makedirs(runs_dir, exist_ok=True)

device = 0 if torch.cuda.is_available() else "cpu"

try:
    import openvino
    ov_version = openvino.get_version()
except Exception:
    ov_version = "unknown"
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| openvino", ov_version)
print("CPU threads used by PyTorch:", torch.get_num_threads())
print("device:", device)

## 1. Build the dataset

Finds every labelled image in every source, removes duplicates with a perceptual hash (dHash), and splits the rest 70/20/10, keeping related photographs in the same split. Needs `imagehash`.

In [ ]:
# Step 1: all source folders
BASE = Path("/mnt/LinuxBackup2/Datasets_of_interest/Marine/Penguin")
OUT = Path(dataset_dir)

sources = [
    BASE / "african-penguin-inaturalist-sam-veryclean",
    BASE / "humboldt-inaturalist-sam",
    BASE / "magellanicus-naturalist-sam",
    BASE / "galapagos-inaturalist-sam",
    BASE / "RoboflowDatasets/SEGMENT/penguin-seg-combined",
    BASE / "RoboflowDatasets/SEGMENT/KIT315-sam",
    BASE / "RoboflowDatasets/SEGMENT/marine-and-aerial-segmentation-martin-sam",
    BASE / "RoboflowDatasets/SEGMENT/penguin-finder-marine-aerial-sam",
    BASE / "RoboflowDatasets/SEGMENT/MyFirstProject2-sam",
    BASE / "RoboflowDatasets/SEGMENT/marine-and-aerial-segmentation-sam",
    BASE / "RoboflowDatasets/SEGMENT/penguin-instance-sam",
    BASE / "RoboflowDatasets/SEGMENT/penguin-seg-sam",
    BASE / "RoboflowDatasets/SEGMENT/penguin-watch/geora-sam",
    BASE / "RoboflowDatasets/det/penguin_3single-sam",
    BASE / "RoboflowDatasets/det/penguin-water-sam",
    BASE / "RoboflowDatasets/det/penguin-water-hard-fp",
    BASE / "RoboflowDatasets/det/OCAT-sam",
    BASE / "RoboflowDatasets/det/pingwiny-spheniscus",
    BASE / "RoboflowDatasets/det/pingwiny-spheniscus2",
    BASE / "RoboflowDatasets/SEGMENT/multiclass/PenguinSpecies-Single-sam",
    BASE / "RoboflowDatasets/SEGMENT/original_splits/Penguin-Seaworld-sam3",
]

In [ ]:
# Step 2: find every image that has a label file
# We look inside every source folder, and inside its train, valid and test folders
# Then collect every image that has a matching label file. 
# At this point, ignore the split an image was in before.. 
# All images are combined and dumped in a big pool, and later in the code (Section 4), a new 70/20/10 split is made


all_images = []   # each entry: (source name, image path, label path)

for src in sources:                     #go through each dataset folder 
    if not src.exists():                #skip a source if its folder is missing
        print("missing:", src)          
        continue
    count = 0                           #how many usable images the source has 
    for split in ["train", "valid", "test"]:
        # src/split/"images" builds a path like .../penguin-water-sam/train/images.
        # .glob("*") lists everything inside the folder
        # If a source doesn't have a "valid" or "test" folder, glob returns nothing
        # So sources with only a train folder still work
        # sorted() puts the files in a fixed order, so every run sees them in the same order 
        # This matters later for when duplicates are removed: The first copy found is kept, 
        # so a fixed order means the same copy is kept each and every time. 
        for img in sorted((src / split / "images").glob("*")): 
            lbl = src / split / "labels" / (img.stem + ".txt")
            if img.suffix.lower() in [".jpg", ".jpeg", ".png"] and lbl.exists():
                all_images.append((src.name, img, lbl))
                count += 1
    print(src.name, count)

print("Total:", len(all_images))

In [ ]:
# Step 3: Remove duplicates
# The same photo can appear in more than one source, for example when a Roboflow 
# dataset was exported twice, or when one dataset is combined into another. 
# Training and testing on copies of the SAME photo would make results look better than they really are, 
# So it is best to keep only one copy of each photo. 

# dHash ("Difference Hash") gives each picture a short fingerprint based on what it 
# LOOKS like, not on exact bytes. A copy that was resized or saved again
# Still looks the same, so it gets the SAME fingerprint and is recognised as a duplicate.
  
seen = set()            # a breadcrumb for fingerprints already seen 
unique_images = []      # The images that are kept 

for source, img, lbl in all_images:             #go through the pool of imgs from the previous cell
    try:
        # Open the image and work out its fingerprint 
        # "with" closes the file again afterwards, so thousands of files are 
        # never left open at once .. open&close

        with Image.open(img) as im:
            # imagehash.dhash() returns a 64-bit hash object 
            # str() turns it into 16 hexadecimal characters, e.g. "f0e4c2d7a1b3c5e9"
            # this is stored in the set and compared.
            fingerprint = str(imagehash.dhash(im))
    except Exception:
        # a damaged or unreadable file/img would crash the loop, so it is skipped
        print("could not open:", img)
        continue

    if fingerprint in seen:         # this would be a picture already seen, so SKIP!
        continue
    seen.add(fingerprint)           # first time fingerprint is being seen
    unique_images.append((source, img, lbl)) #remember the source and keep it 

#How many copies were dropped and how many images remain. 
print("Duplicates removed:", len(all_images) - len(unique_images))
print("Images left:", len(unique_images))


# img is shrunk to 9x8 pixels. All the detail disappears and only the rough pattern of light and dark is left
# turn it grey, colour is ignored
# compare neighbours. In each row, check whether each pixel is brighter than the one to its right. 
# this gives 8 comparisons per row * 8 rows= 64 yes/no answers
# These 64 answers are the fingerprint  

**Split the Dataset 70/20/10**

In [ ]:
# STEP 4: Split the images 70/20/10, keeping related photos together.
# If two nearly identical photos end up in BOTH training and test, the model is
# tested on something it has practically already seen, and the results look
# better than they really are. So related photos are put into a "group", and
# each group goes entirely into one split.
#  - GEORa is one time-lapse camera, so its frames are near-identical:
#    all of them go to training.
#  - iNaturalist files are named <observation>_<photo>.jpg: all photos from one
#    observation (same person, same place, same moment) form one group.
#  - Roboflow files are named <original>_jpg.rf.<hash>.jpg: all copies made from
#    one original image form one group.

def group_key(source, img):
    #Return a label saying which group this image belongs to.
    # Roboflow copy, e.g. "IMG_1234_jpg.rf.abc123.jpg"
    if ".rf." in img.name:
        # cut at ".rf."      -> "IMG_1234_jpg"
        # drop the last "_jpg" -> "IMG_1234", the original file's name
        original = img.name.split(".rf.")[0].rsplit("_", 1)[0]
        return "roboflow:" + original

    # iNaturalist photo, e.g. "61280804_98046292.jpg"
    # img.stem = "61280804_98046292", split at "_" -> ["61280804", "98046292"]
    parts = img.stem.split("_")   # the key step for iNaturalist names
    if len(parts) == 2 and parts[0].isdigit() and parts[1].isdigit():
        return "inaturalist:" + parts[0]       # the observation number

    # Any other name, e.g. "frame_001.png": the image is a group on its own.
    # The source name is added so equal names in different sources stay apart.
    return source + ":" + img.stem


# Separate GEORa from everything else. x[0] is the source name of each entry.
geora = [x for x in unique_images if x[0] == "geora-sam"]
rest = [x for x in unique_images if x[0] != "geora-sam"]

# Build the groups: a dictionary from group label to the list of its images.
# setdefault() creates an empty list the first time a label appears,
# then adds the image to that group's list.
groups = {}
for x in rest:
    key = group_key(x[0], x[1])
    groups.setdefault(key, []).append(x)

# Shuffle the GROUPS (not single images) into a random order.
# Sorting first, then shuffling with a fixed seed (42), gives exactly the
# same "random" order every time the notebook runs, so the split can be repeated.
keys = sorted(groups)
random.seed(42)
random.shuffle(keys)

# Fill the splits group by group, until each holds its share of the images.
train, valid, test = list(geora), [], []   # training starts with all GEORa images
n = len(rest)        # images to split (GEORa not included)
done = 0             # images placed so far
for key in keys:
    if done < 0.7 * n:              # first 70% of the images -> training
        train += groups[key]
    elif done < 0.9 * n:            # next 20% -> validation
        valid += groups[key]
    else:                           # last 10% -> test
        test += groups[key]
    done += len(groups[key])        # count IMAGES, not groups, so the shares
                                    # are 70/20/10 of images, as the thesis says

print("groups:", len(groups), "from", n, "images (GEORa excluded)")
print("train:", len(train), "(GEORa:", len(geora), ") valid:", len(valid), "test:", len(test))

In [ ]:
# STEP 5: Copy the images and labels into the new dataset folder.
# Every file gets a NEW NAME made of its source and a running number,
# e.g. "african-penguin-inaturalist-sam-veryclean_000123.jpg".
# Two different sources can contain files with the same name (e.g. "IMG_0001.jpg").
# With new, unique names, one file can never overwrite another.

# Stop straight away if the output folder already exists. Copying into an old
# folder would mix new files with leftovers from an earlier run.
assert not OUT.exists(), "The output folder already exists. Delete it first, then run again."

number = 0   # one counter for ALL splits, so every file in the dataset has a unique number

for split_name, images in [("train", train), ("valid", valid), ("test", test)]:
    # Make the YOLO folder layout: <split>/images and <split>/labels.
    # parents=True also creates the folders above them (OUT and OUT/train) if needed.
    (OUT / split_name / "images").mkdir(parents=True)
    (OUT / split_name / "labels").mkdir(parents=True)

    for source, img, lbl in images:
        # {number:06d} writes the number with 6 digits, padded with zeros:
        # 123 -> "000123". This keeps the files in order when sorted by name.
        new_name = f"{source}_{number:06d}"

        # Copy the image. .lower() turns ".JPG" into ".jpg", so all extensions match.
        shutil.copy(img, OUT / split_name / "images" / (new_name + img.suffix.lower()))

        # Copy its label, with the SAME new name, so YOLO can pair them up
        # (image "abc_000123.jpg" goes with label "abc_000123.txt").
        shutil.copy(lbl, OUT / split_name / "labels" / (new_name + ".txt"))

        number += 1

# Write data.yaml, the file that tells Ultralytics where everything is.
#  train/val/test: the image folders. Ultralytics finds each label automatically,
#                  by swapping "images" for "labels" in the path.
#  nc: 1           there is one class...
#  names           ...and it is called "penguin".
yaml = f"""train: {OUT}/train/images
val: {OUT}/valid/images
test: {OUT}/test/images
nc: 1
names: ['penguin']
"""
(OUT / "data.yaml").write_text(yaml)
print("Done:", OUT)

In [ ]:
# summary of each split
for split_name in ["train", "valid", "test"]:
    label_files = list((OUT / split_name / "labels").glob("*.txt"))
    penguins = 0
    empty = 0
    for f in label_files:
        boxes = [line for line in f.read_text().splitlines() if line.strip()]
        penguins += len(boxes)
        if len(boxes) == 0:
            empty += 1
    print(f"{split_name}: {len(label_files)} images, {penguins} penguins, {empty} images with no penguins")

In [ ]:
# list the African Penguin test images, so results can also be reported for them alone
african = sorted((OUT / "test" / "images").glob("african-penguin-inaturalist-sam-veryclean_*"))
(OUT / "test_african.txt").write_text("\n".join(str(p) for p in african) + "\n")
print("African Penguin test images:", len(african))

## 2. Training

All four models use the same settings, so the only differences between them are model size and YOLO version: SGD (Stochastic Gradient Descent) with the Ultralytics default learning rate, momentum and weight decay, the default augmentation, batch size 16, and seed 42 with deterministic training. The exact values are saved in each run's `args.yaml`.

YOLOv11n was trained a second time from a clean folder. With the same seed, the second run reproduced the first exactly.

In [ ]:
# settings shared by all four models
train_settings = dict(
    data=data_yaml,
    epochs=100,
    imgsz=640,
    batch=16,
    workers=8,
    optimizer="SGD",    # Stochastic Gradient Descent
    seed=42,
    deterministic=True,
    patience=20,        # stop early after 20 epochs without improvement
    device=device,
    project=runs_dir,
    exist_ok=True,      # keeps folder names fixed (runs_702010/yolov11n, ...). Re-running overwrites them.
    plots=True,
)

In [ ]:
# YOLOv11n
model = YOLO("yolo11n.pt")   # COCO-pretrained weights, all layers trainable
model.train(name="yolov11n", **train_settings)

In [ ]:
# YOLOv11m
model = YOLO("yolo11m.pt")
model.train(name="yolov11m", **train_settings)

In [ ]:
# YOLOv26n
model = YOLO("yolo26n.pt")
model.train(name="yolov26n", **train_settings)

In [ ]:
# YOLOv26m
model = YOLO("yolo26m.pt")
model.train(name="yolov26m", **train_settings)

## 3. Experiment 1.2: detection accuracy

Each model is exported to OpenVINO at FP32 and evaluated on the CPU, on the full test set and on its African Penguin photographs.

In [ ]:
# The four trained models, by name, with the path to each one's best weights
MODELS = {
    "YOLOv11n": runs_dir + "/yolov11n/weights/best.pt",
    "YOLOv26n": runs_dir + "/yolov26n/weights/best.pt",
    "YOLOv11m": runs_dir + "/yolov11m/weights/best.pt",
    "YOLOv26m": runs_dir + "/yolov26m/weights/best.pt",
}
# Check every weights file exists before running long experiments on it
for name, weights in MODELS.items():
    print(name, "found" if os.path.exists(weights) else "MISSING", weights)

# A second data file, identical to data.yaml except for one line:
# its "test" entry points to test_african.txt INSTEAD OF the whole test folder.
# Ultralytics accepts a .txt list of image paths as a test set, so when this file
# is used, only the 835 African Penguin photos are evaluated!
african_yaml = dataset_dir + "/data_test_african.yaml"
with open(african_yaml, "w") as f:
    f.write(f"train: {dataset_dir}/train/images\n")   # not used for testing, but required
    f.write(f"val: {dataset_dir}/valid/images\n")     # not used for testing, but required
    f.write(f"test: {dataset_dir}/test_african.txt\n")  # the only line that differs
    f.write("nc: 1\nnames: ['penguin']\n")
print("African test list exists:", os.path.exists(dataset_dir + "/test_african.txt"))

In [ ]:
# Experiment 1.2: detection accuracy
# Each model is exported to OpenVINO (FP32) and evaluated on the CPU, so these are the
# accuracies of the deployed models. mAP does not depend on the confidence threshold.

detection_rows = []         # one row of results for each model per test set 
for name, weights in MODELS.items():
    # convert the trained PyTorch model (best.pt) into OpenVINO's format
    # format = "openvino":  OpenVINO Intermediate Representation (a .xml + .bin pair)
    # imgsz = 640:          The model expects 640 x 640 images
    # dynamic = False       The input size is FIXED at 640 x 640 
    # half = False          Keep the full 32-bit precision FP32.. not FP16
    # export()              Returns the path of the new folder. e.g. .../weights/best_openvino_model

    ov_path = YOLO(weights).export(format="openvino", imgsz=640, dynamic=False, half=False)

    # load the exported model. task = "detect" tells Ultralytics it is a detection 
    # model, which it cannot always work out from an exported folder. 
    ov_model = YOLO(ov_path, task="detect")

    # TEST it twice. on the whole test set and then on Afr. Penguins only 
    for test_set, yaml_path in [("all", data_yaml), ("african", african_yaml)]: #this is what comes out in the .csv file

        #val() runs the model on every test image, compares its boxes with the labels, and calculates the accuracy metrics
        # split = "test":       use the "test" entry of the data file
        # device = "intel:cpu"  run it with OpenVINO on the Intel CPU


        #PRECISION: of all the boxes the model drew, what is the share that were real penguins
        #RECALL: of all the real penguins, how many did the model actually find 
        m = ov_model.val(data=yaml_path, split="test", imgsz=640, device="intel:cpu", verbose=False)
        row = {"model": name, "test_set": test_set,
               "map50": round(m.box.map50, 4), "map50_95": round(m.box.map, 4),
               "precision": round(m.box.mp, 4), "recall": round(m.box.mr, 4)} #mean precision and mean recall 
        print(row)
        detection_rows.append(row)

# Save all the rows to a .csv file. DictWriter turns each dictionary into one line.. 
# and uses the keys as the column heads 
with open(results_dir + "/detection_702010.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(detection_rows[0].keys()))
    writer.writeheader()
    writer.writerows(detection_rows)
print("Saved detection_702010.csv")

**Why doesn't mAP depend on the confidence threshold?**  
Every box comes with a confidence score. Raising the threshold gives fewer boxes: higher precision, lower recall. Lowering it does the opposite.  
To calculate mAP, Ultralytics doesn't pick one threshold. It tries all of them, from very low to very high and draws the precision recall curve. AP (Average Precision) is the area under the curve. Since it covers every threshold, no single choice of threshold affects it. 

Precision and recall are taken at one point on the curve: The confidence that gives the BEST balance between them (the highest F1).  

## 4. Experiment 1.2: counting accuracy

In [ ]:
# Experiment 1.2: counting accuracy (predictions above confidence 0.25)
# - MAE, within 5% and the total count use photos that contain penguins.
# - Photos with no penguins are used to count false detections.

def true_count(img_path):
    # the real number of penguins in a photo = the number of lines in its label file
    # turn the image into its label path
    # ../test/images/abc_000123.jpg  ->  .../test/labels/abc_000123.txt
    #.replace() will swap the folder name. rsplit(".", 1)[0] removes the ".jpg" extension.
    lbl = img_path.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"
    with open(lbl) as f:
        # each penguin is one line. line.strip() ignores blank lines so an empty label file counts as 0 penguins
        return sum(1 for line in f if line.strip())

test_images = sorted(glob.glob(dataset_dir + "/test/images/*")) # all 3,383 test photos 

counting_rows = []

for name, weights in MODELS.items():
    model = YOLO(weights)   # the trained PyTorch model (best.pt)

    # Run the model once on every test photo and store the results
    # Both test sets ("all" and "African") are then worked out from the same list so that nothing is predicted twice
    per_image = []   # (is African Penguin photo, true count, predicted count)

    for img in test_images:
        # conf= 0.25: only boxes with a conf of 0.25 or more are counted
        # [0] takes the result for this one photo, boxes are its predicted boxes 
        # An Afr. Penguin photo is recognised by its file name prefix 

        predicted = len(model(img, conf=0.25, verbose=False)[0].boxes)
        is_african = os.path.basename(img).startswith("african-penguin") 
        per_image.append((is_african, true_count(img), predicted))

    for test_set in ["all", "african"]:
        # "all" keeps every photo. "african" keeps a photo only if r[0] is true
        rows = [r for r in per_image if test_set == "all" or r[0]]
        with_penguins = [r for r in rows if r[1] > 0] # r[1] is the true count
        empty = [r for r in rows if r[1] == 0]        # photos with no penguins 

        # error per photo: how far off the count was, ignoring over or under 
        # "_" means "a value we do not need here" (the is_african flag)
        errors = [abs(pred - true) for _, true, pred in with_penguins]

        # photos counted within 5% of their true count 
        # only photos with penguins, so we never divide by a true count of 0
        within5 = sum(1 for _, true, pred in with_penguins if abs(pred - true) / true <= 0.05)

        # Total over the WHOLE TEST SET, for the "census level target" defined in thesis
        total_true = sum(true for _, true, _ in with_penguins) # all real penguins, e.g. 15,632
        total_pred = sum(pred for _, _, pred in with_penguins) # all counted penguins

        row = {
            "model": name, "test_set": test_set,
            "photos_with_penguins": len(with_penguins),
            "mae": round(sum(errors) / len(errors), 2),         # average error per photo
            "within_5pct": round(100 * within5 / len(with_penguins), 1), #e.g. 68.1% of photos passed.. % of photos within 5%
            "total_true": total_true,
            "total_predicted": total_pred,
            # signed error: negative means undercounted
            "total_error_pct": round(100 * (total_pred - total_true) / total_true, 2), # e.g. -1.1%
            "empty_photos": len(empty),
            "false_boxes_on_empty": sum(pred for _, _, pred in empty),
            "empty_photos_with_a_false_box": sum(1 for _, _, pred in empty if pred > 0),
        }
        print(row)
        counting_rows.append(row)

#save all rows to counting_702010.csv 
with open(results_dir + "/counting_702010.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(counting_rows[0].keys()))
    writer.writeheader()
    writer.writerows(counting_rows)
print("Saved counting_702010.csv")

## 5. Experiment 2: CPU latency

Both runtimes are timed in the same session on the same images. Every test image is first resized and padded to 640 x 640, because the OpenVINO export has a fixed 640 x 640 input. Each timing includes image loading, pre-processing, inference and post-processing, at batch size 1 after 10 warm-up images.

In [ ]:
# make 640 x 640 copies of the test images, to time latency
# OpenVINO models have a fixed 640 x 640 input.. but PyTorch pads each photo
# only as much as it needs (a 4:3 photo becomes 640 x 480). PyTorch would then process 
# about a third fewer pixels which is an unfair advantage. Giving both runtimes the same input size makes timing 
# comparison fair. These copies are only used for timing. 
# Accuracy is always measured on the original photos. 

square_dir = results_dir + "/test_640_702010"
os.makedirs(square_dir, exist_ok=True)      # create the folder; no error if it already exists

square_images = []                          # path to the 640 x 640 copies 
for img in test_images:     
    out = square_dir + "/" + os.path.basename(img)      # same file name, in the new folder

    if not os.path.exists(out):                         # done once: skip photos that have already been converted
        im = cv2.imread(img)                            # read the photos as a grid of pixels
        h, w = im.shape[:2]                             # its height and width in pixels 

        # shrink or enlarge the img so the longer side becomes exactly 640, 
        # keeping the shape of the photo so the penguins are not stretched
        scale = 640 / max(h, w)             
        im = cv2.resize(im, (round(w * scale), round(h * scale)))

        # make a 640 x 640 grey square. 114 is the shade of grey Ultralytics uses for padding 
        # (640, 640, 3) = 640 rows, 640 columns, 3 colour channels (RGB colour).
        canvas = np.full((640, 640, 3), 114, dtype=np.uint8)   # grey padding, like Ultralytics, each pixel val is a whole number from 0 to 255
        canvas[:im.shape[0], :im.shape[1]] = im
        cv2.imwrite(out, canvas)
    square_images.append(out)
print(len(square_images), "images ready in", square_dir)

In [ ]:
# Time PyTorch and OpenVINO on the CPU, one straight after the other.
# Running both in the same session, on the same 640 x 640 images, means any change
# in the computer's speed during the day affects both runtimes equally.

def time_model(model, device):
    """Return the mean and standard deviation of the time per image, in milliseconds."""

    # Warm-up: the first few predictions are always slow, because the model is still
    # being loaded into memory, and OpenVINO is still compiling it for this CPU.
    # These 10 runs are NOT timed, so they do not distort the average.
    for img in square_images[:10]:
        model(img, imgsz=640, device=device, verbose=False)

    times = []
    for img in square_images:              # all 3,383 test images
        start = time.perf_counter()        # a precise stopwatch: start
        # One full prediction: read the image from disk, prepare it, run the model,
        # then tidy up its output (including NMS for YOLOv11). All of it is timed,
        # because a field user waits for all of it.
        model(img, imgsz=640, device=device, verbose=False)
        # stop the stopwatch; perf_counter() is in seconds, so * 1000 gives milliseconds
        times.append((time.perf_counter() - start) * 1000)

    mean = sum(times) / len(times)         # average time per image
    # Standard deviation: how much the time varies from image to image.
    # (Each time's distance from the mean, squared, averaged, then square-rooted.)
    std = (sum((t - mean) ** 2 for t in times) / len(times)) ** 0.5
    return mean, std


latency_rows = []
for name, weights in MODELS.items():
    # The OpenVINO version was already exported in section 3; this finds its folder
    ov_path = weights.replace("best.pt", "best_openvino_model")

    # Time the same model twice, back to back:
    #  "PyTorch":  the original best.pt, forced onto the CPU with device="cpu"
    #              (otherwise it would use the GPU)
    #  "OpenVINO": the exported model, on OpenVINO's CPU device ("intel:cpu")
    for runtime, model, dev in [("PyTorch", YOLO(weights), "cpu"),
                                ("OpenVINO", YOLO(ov_path, task="detect"), "intel:cpu")]:
        mean, std = time_model(model, dev)
        row = {"model": name, "runtime": runtime, "precision": "FP32",
               "mean_ms": round(mean, 2), "std_ms": round(std, 2),
               "fps": round(1000 / mean, 2),      # images per second = 1000 ms / time per image
               "n_images": len(square_images)}
        print(row)
        latency_rows.append(row)

# Save to latency_702010.csv, the source of Table 4.4
with open(results_dir + "/latency_702010.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(latency_rows[0].keys()))
    writer.writeheader()
    writer.writerows(latency_rows)
print("Saved latency_702010.csv")

## 6. Experiment 3: FP32 against FP16

Exports each model to OpenVINO a second time with FP16 weights, and compares model size, accuracy and latency with FP32 in the same session.

In [ ]:
# Experiment 3: OpenVINO FP32 against FP16 (model size, accuracy and speed).
# FP32 stores each of the model's numbers (weights) in 32 bits; FP16 uses 16 bits.
# The question: does halving the storage change accuracy or speed on this CPU?

def folder_size_mb(path):
    """Total size of every file in an exported model folder, in megabytes."""
    total = 0
    # os.walk visits the folder and any folders inside it, listing their files.
    # An OpenVINO model folder holds a .xml file (the model's structure),
    # a .bin file (the weights, almost all of the size) and a small metadata file.
    for root, dirs, files in os.walk(path):
        for f in files:
            total += os.path.getsize(os.path.join(root, f))   # size of each file, in bytes
    return round(total / 1e6, 1)                              # bytes -> megabytes


fp16_rows = []
for name, weights in MODELS.items():
    # Reuse the FP32 export from section 3. Export it again only if it is missing.
    fp32_dir = weights.replace("best.pt", "best_openvino_model")
    if not os.path.exists(fp32_dir):
        fp32_dir = YOLO(weights).export(format="openvino", imgsz=640, dynamic=False, half=False)

    # Ultralytics names the export folder after the weights file:
    # best.pt -> best_openvino_model. Exporting FP16 from best.pt would therefore
    # OVERWRITE the FP32 folder. Exporting from a copy called best_fp16.pt gives it
    # its own folder, best_fp16_openvino_model, so both versions exist side by side.
    fp16_weights = weights.replace("best.pt", "best_fp16.pt")
    shutil.copy(weights, fp16_weights)
    # half=True: store the weights in 16 bits (FP16) instead of 32 bits
    fp16_dir = YOLO(fp16_weights).export(format="openvino", imgsz=640, dynamic=False, half=True)

    # Measure both versions in exactly the same way, in this same session
    for precision, folder in [("FP32", fp32_dir), ("FP16", fp16_dir)]:
        model = YOLO(folder, task="detect")
        # accuracy on the full test set and on the African Penguin photos
        m_all = model.val(data=data_yaml, split="test", imgsz=640, device="intel:cpu", verbose=False)
        m_african = model.val(data=african_yaml, split="test", imgsz=640, device="intel:cpu", verbose=False)
        # speed, using the same timing function and 640 x 640 images as Experiment 2
        mean, std = time_model(model, "intel:cpu")
        row = {
            "model": name, "precision": precision, "size_mb": folder_size_mb(folder),
            "map50_all": round(m_all.box.map50, 4), "map50_95_all": round(m_all.box.map, 4),
            "map50_african": round(m_african.box.map50, 4), "map50_95_african": round(m_african.box.map, 4),
            "mean_ms": round(mean, 2), "std_ms": round(std, 2), "fps": round(1000 / mean, 2),
            "n_images": len(square_images),
        }
        print(row)
        fp16_rows.append(row)

# Save to fp16_702010.csv, the source of Table 4.5 and Appendix Table A.1
with open(results_dir + "/fp16_702010.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(fp16_rows[0].keys()))
    writer.writeheader()
    writer.writerows(fp16_rows)
print("Saved fp16_702010.csv")

## 7. Experiment 4.2: hyperparameter tuning

The settings below match those recorded in the tuning run's `args.yaml`.

In [ ]:
# Experiment 4.2: 10 Ray Tune trials with Optuna, 20 epochs each, continuing from the trained
# YOLOv11n. yolo_any_tune.py and bench_schema.py (written by Prof. Dane Brown) are in ../Tools.
!python3 ../Tools/yolo_any_tune.py \
    --data /home/hluma/penguin_dataset_702010 \
    --model /home/hluma/hluma_results/runs_702010/yolov11n/weights/best.pt \
    --task detect --tune --tune_iter 10 --epochs 20 --batch_sz 16

## 8. Experiment 5: scale augmentation

Retrains YOLOv11n with a narrower scale range, keeping every other setting the same, and evaluates it as in Experiment 1.2.

In [ ]:
# Experiment 5: does a narrower scale augmentation help YOLOv11n?
# During training, Ultralytics randomly zooms each training image in or out.
# The "scale" setting controls how much:
#   scale=0.5 (default): zoom between 0.5x and 1.5x
#   scale=0.3 (tested):  zoom between 0.7x and 1.3x
# Zooming teaches the model to find penguins at different sizes.
# This experiment retrains YOLOv11n with the narrower range and compares it with
# the baseline. Every other setting is identical, so scale is the ONLY difference.

# The baseline already exists from section 2 (trained with the default scale=0.5)
scale_models = {"0.5 (default)": runs_dir + "/yolov11n/weights/best.pt"}

# Train a new YOLOv11n from the COCO weights, exactly like the baseline, except scale.
# **train_settings unpacks the shared settings dictionary (epochs, batch, seed 42,
# SGD and so on) into keyword arguments, so nothing else can differ by accident.
model = YOLO("yolo11n.pt")
model.train(name="yolov11n_scale03", scale=0.3, **train_settings)
scale_models["0.3"] = runs_dir + "/yolov11n_scale03/weights/best.pt"

scale_rows = []
for scale, weights in scale_models.items():   # evaluate the baseline AND the new model
    # Detection accuracy: the same method as Experiment 1.2 (OpenVINO FP32 on the CPU)
    ov_path = YOLO(weights).export(format="openvino", imgsz=640, dynamic=False, half=False)
    ov_model = YOLO(ov_path, task="detect")
    m_all = ov_model.val(data=data_yaml, split="test", imgsz=640, device="intel:cpu", verbose=False)
    m_african = ov_model.val(data=african_yaml, split="test", imgsz=640, device="intel:cpu", verbose=False)

    # Counting accuracy: the same method as Experiment 1.2 (confidence 0.25),
    # written as running totals instead of storing every photo
    model = YOLO(weights)
    errors, within5, total_true, total_pred, false_on_empty = [], 0, 0, 0, 0
    for img in test_images:
        true = true_count(img)                                      # real penguins
        pred = len(model(img, conf=0.25, verbose=False)[0].boxes)   # counted penguins
        if true == 0:                     # a photo with no penguins:
            false_on_empty += pred        # every box on it is a false detection
            continue                      # and it is left out of the other measures
        errors.append(abs(pred - true))   # this photo's counting error
        # A comparison gives True (1) or False (0), so this adds 1 for every photo
        # counted within 5% of its true count
        within5 += abs(pred - true) / true <= 0.05
        total_true += true                # running totals for the whole test set
        total_pred += pred

    row = {
        "model": "YOLOv11n", "scale": scale,
        "map50_all": round(m_all.box.map50, 4), "map50_95_all": round(m_all.box.map, 4),
        "map50_african": round(m_african.box.map50, 4), "map50_95_african": round(m_african.box.map, 4),
        "mae": round(sum(errors) / len(errors), 2),
        "within_5pct": round(100 * within5 / len(errors), 1),
        "total_error_pct": round(100 * (total_pred - total_true) / total_true, 2),
        "false_boxes_on_empty": false_on_empty,
    }
    print(row)
    scale_rows.append(row)

# Save to scale_702010.csv, the source of Table 4.8 and Appendix Table A.4
with open(results_dir + "/scale_702010.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(scale_rows[0].keys()))
    writer.writeheader()
    writer.writerows(scale_rows)
print("Saved scale_702010.csv")

## 9. Figures and the density split

`make_thesis_figures.py` draws the accuracy-latency plot, the training curves, the tuning-trial figure, the counting results by number of penguins per photograph (`--density`) and example detections (`--examples`).

In [ ]:
# Thesis figures, the density split and the example detections (see make_thesis_figures.py)
!python3 make_thesis_figures.py --all